In [1]:
from datasets import load_dataset
data = load_dataset("batuhanmtl/job_resume_fit")

README.md:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

job_resume_fit.csv:   0%|          | 0.00/30.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2385 [00:00<?, ? examples/s]

In [2]:
df = data["train"].to_pandas()

In [ ]:
import pandas as pd

In [ ]:
df.head(1)

,ID,resume_text,job_text,category,job_required_skills,resume_skill_list,ai_matched_skills,ai_match_score,skill_string_match_score,fuzzy_match_score
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,Human Resources Business Partner\n------------...,HR,"['human resources management', 'talent acquisi...","['customer service', 'team management', 'marke...","['human resources management', 'employee relat...",49.4,12.5,63.24


In [ ]:
df.tail()

,ID,resume_text,job_text,category,job_required_skills,resume_skill_list,ai_matched_skills,ai_match_score,skill_string_match_score,fuzzy_match_score
2380,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,Aviation Maintenance Manager\n----------------...,AVIATION,['FAA Airframe and Powerplant (A&P) certificat...,"['logistics', 'inventory control', 'customer s...","['FAA regulations', 'quality assurance', 'main...",46.84,12.12,57.82
2381,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...",Aviation Maintenance Manager\n----------------...,AVIATION,['FAA Airframe and Powerplant (A&P) certificat...,"['arbitration', 'budgets', 'continuous improve...","['budget management', 'maintenance programs', ...",25.18,12.12,54.06
2382,31605080,GEEK SQUAD AGENT Professional...,Aviation Maintenance Manager\n----------------...,AVIATION,['FAA Airframe and Powerplant (A&P) certificat...,"['windows', 'mac', 'ios', 'android', 'technica...","['aircraft maintenance', 'troubleshooting tech...",13.01,6.06,51.19
2383,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,Aviation Maintenance Manager\n----------------...,AVIATION,['FAA Airframe and Powerplant (A&P) certificat...,"['sage fundraising 50', 'microsoft word', 'mic...","['training coordination', 'leadership', 'perso...",17.79,6.06,53.67
2384,37473139,STOREKEEPER II Professional Sum...,Aviation Maintenance Manager\n----------------...,AVIATION,['FAA Airframe and Powerplant (A&P) certificat...,"['autocad', 'excel', 'microsoft office', 'powe...","['aircraft maintenance', 'quality assurance', ...",40.15,9.09,54.58


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2385 entries, 0 to 2384
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        2385 non-null   int64  
 1   resume_text               2385 non-null   object 
 2   job_text                  2385 non-null   object 
 3   category                  2385 non-null   object 
 4   job_required_skills       2385 non-null   object 
 5   resume_skill_list         2385 non-null   object 
 6   ai_matched_skills         2385 non-null   object 
 7   ai_match_score            2385 non-null   float64
 8   skill_string_match_score  2385 non-null   float64
 9   fuzzy_match_score         2385 non-null   float64
dtypes: float64(3), int64(1), object(6)
memory usage: 186.5+ KB


In [ ]:
df.describe()

,ID,ai_match_score,skill_string_match_score,fuzzy_match_score
count,2.385000e+03,2385.000000,2385.00000,2385.000000
mean,3.189812e+07,44.035862,11.55426,56.579698
std,2.151701e+07,24.628549,7.48703,6.778110
min,3.547447e+06,0.000000,0.00000,26.150000
25%,1.757603e+07,22.230000,6.06000,52.030000
50%,2.521301e+07,40.650000,10.00000,56.400000
75%,3.620648e+07,63.950000,15.79000,61.070000
max,9.971441e+07,100.000000,41.67000,84.410000


In [ ]:
df.isnull().sum()

,0
ID,0
resume_text,0
job_text,0
category,0
job_required_skills,0
resume_skill_list,0
ai_matched_skills,0
ai_match_score,0
skill_string_match_score,0
fuzzy_match_score,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.shape

(2385, 10)

In [ ]:
df = df[['resume_text','job_text','category','job_required_skills','resume_skill_list','ai_match_score']]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
resume_tfidf = TfidfVectorizer(stop_words='english',max_features=2000)
resume = resume_tfidf.fit_transform(df['resume_text'])

In [ ]:
job_tfidf = TfidfVectorizer(stop_words='english',max_features=2000)
job = job_tfidf.fit_transform(df['job_text'])

In [ ]:
skill_tfidf = TfidfVectorizer(stop_words='english',max_features=1000)
required_skill = skill_tfidf.fit_transform(df['job_required_skills'])

In [ ]:
resume_skill_tfidf = TfidfVectorizer(stop_words='english',max_features=1000)
resume_skill = resume_skill_tfidf.fit_transform(df['resume_skill_list'])

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore')
category = encoder.fit_transform(df[['category']])

In [ ]:
from scipy.sparse import hstack
X = hstack([resume,job,required_skill,resume_skill,category])
y = df['ai_match_score']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import joblib
joblib.dump(resume_tfidf, "resume_tfidf.pkl")
joblib.dump(job_tfidf, "job_tfidf.pkl")
joblib.dump(skill_tfidf, "skill_tfidf.pkl")
joblib.dump(resume_skill_tfidf, "resume_skill_tfidf.pkl")
joblib.dump(encoder, "category_encoder.pkl")

['category_encoder.pkl']

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [ ]:
rf = RandomForestRegressor(n_estimators=300,max_depth=30,min_samples_split=5,min_samples_leaf=2,max_features='sqrt',random_state=42,n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [ ]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

In [ ]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train_dense, y_train)
gb_pred = gb.predict(X_test_dense)

In [ ]:
import numpy as np
def evaluate_model(model, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print("="*50)
    print(model)
    print("="*50)
    print("MAE :", round(mae,4))
    print("MSE :", round(mse,4))
    print("RMSE:", round(rmse,4))
    print("R2  :", round(r2,4))
    print()

In [ ]:
evaluate_model("Decision Tree",y_test,dt_pred)

Decision Tree
MAE : 14.5445
MSE : 366.078
RMSE: 19.1332
R2  : 0.3946



In [ ]:
evaluate_model("Random forest",y_test,rf_pred)

Random forest
MAE : 10.3617
MSE : 181.1046
RMSE: 13.4575
R2  : 0.7005



In [ ]:
evaluate_model("Gradient boosting",y_test,gb_pred)

Gradient boosting
MAE : 10.8662
MSE : 199.728
RMSE: 14.1325
R2  : 0.6697



In [ ]:
models = pd.DataFrame({
    "Model":["Decision Tree","Random Forest","Gradient Boosting"],
    "R2 Score":[r2_score(y_test,dt_pred),r2_score(y_test,rf_pred),r2_score(y_test,gb_pred)]
})
print(models)

               Model  R2 Score
0      Decision Tree  0.394611
1      Random Forest  0.700505
2  Gradient Boosting  0.669707


In [ ]:
joblib.dump(rf,"resume_match_model.pkl")
print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
from sklearn.metrics import mean_absolute_error

pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, pred)

print(mae)

10.3616513593751
